<a href="https://colab.research.google.com/github/JulTob/Modelling/blob/main/GLM/GLM_2_P1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# GLM · Caso 2 · P1 — Vehículo, zona y siniestros 🏵

**Equipo 07-Cox · Dirección comercial**

> **Pregunta del comité:** «¿Qué clientes dan más siniestros: depende del vehículo, de la zona o de las dos cosas? Si las motos circularan sobre todo por ciudad, ¿cuál de las dos pesaría de verdad?».

Fuente: [Caso 2, Unidad 2.7 · P1](https://asunmayoral.github.io/GLM/caso2/caso2_conteos.html#sec-u27).

Vamos **paso a paso**: (1) comprobamos los datos; (2) construimos una tabla de contingencia de **tres vías**; (3) la convertimos en porcentajes *dentro de cada combinación*; (4) miramos la composición vehículo–zona; (5) contrastamos las asociaciones con modelos log-lineales.

**Unidad estadística:** una póliza. **Respuesta:** `tuvo_siniestro` (sí/no, equivalente a `n_siniestros > 0`). Los conteos de las celdas representan **pólizas**, no la suma del número de partes.

## 0. Cargar y comprobar la cartera 🔰

Utilizamos **la misma cartera del equipo 07** que aparece en el Colab `GLM_2_4`, ahora disponible en la carpeta `GLM/Datos/C2` del repositorio. Solo necesitamos las variables de P1; no cambiaremos ni descartaremos pólizas silenciosamente.

**Condiciones que verificamos:** identificadores únicos, valores presentes, conteos válidos y coherencia entre `tuvo_siniestro` y `n_siniestros`.

In [ ]:
library(tidyverse)

fuente <- "https://raw.githubusercontent.com/JulTob/Modelling/main/GLM/Datos/C2/cartera.csv"
cartera <- readr::read_csv(fuente, show_col_types = FALSE)

variables_p1 <- c("id_poliza", "tipo_vehiculo", "zona_circulacion",
                  "n_siniestros", "antiguedad")
stopifnot(
  all(variables_p1 %in% names(cartera)),
  nrow(cartera) > 0,
  !anyNA(cartera[, variables_p1]),
  dplyr::n_distinct(cartera$id_poliza) == nrow(cartera),
  all(cartera$n_siniestros >= 0),
  all(cartera$n_siniestros == floor(cartera$n_siniestros))
)
if ("tuvo_siniestro" %in% names(cartera)) {
  stopifnot(all(as.integer(cartera$tuvo_siniestro) ==
                as.integer(cartera$n_siniestros > 0)))
}

datos_p1 <- cartera %>%
  transmute(
    id_poliza,
    tipo_vehiculo = factor(tipo_vehiculo),
    zona_circulacion = factor(zona_circulacion),
    tuvo_siniestro = factor(if_else(n_siniestros > 0, "Sí", "No"),
                           levels = c("No", "Sí"))
  )

cat("Pólizas:", nrow(datos_p1), "\n")
glimpse(datos_p1)

## 1. La tabla de contingencia de tres vías ⚙️

Una tabla corriente cruza dos variables. Aquí necesitamos tres índices:

$$
n_{[v,z,e]} = \#\{\text{pólizas de vehículo }v,\text{ zona }z,\text{ evento }e\}
$$

La pregunta exige conservar **simultáneamente** vehículo, zona y presencia de siniestro. Una tabla `tipo_vehiculo × tuvo_siniestro` mezcla las zonas: si los tipos de vehículo no se distribuyen igual entre zonas, puede confundir su efecto aparente.

Mostraremos dos columnas (`No` y `Sí`) para cada pareja **vehículo × zona**.

In [ ]:
tabla_3v <- xtabs(
  ~ tipo_vehiculo + zona_circulacion + tuvo_siniestro,
  data = datos_p1
)

stopifnot(sum(tabla_3v) == nrow(datos_p1))

# Filas: cada pareja vehículo–zona. Columnas: ¿tuvo algún siniestro?
ftable(tabla_3v,
       row.vars = c("tipo_vehiculo", "zona_circulacion"),
       col.vars = "tuvo_siniestro")

## 2. De los recuentos a las proporciones 💡

Para comparar grupos de distinto tamaño, dividimos por el **número de pólizas de esa misma pareja vehículo–zona**:

$$
\widehat p_{[v,z]} =
\frac{n_{[v,z,\mathrm{Sí}]}}{n_{[v,z,\mathrm{Sí}]}+n_{[v,z,\mathrm{No}]}}.
$$

Así, un porcentaje del 40 % significa que, **dentro de ese grupo**, cuatro de cada diez pólizas registraron al menos un siniestro durante su tiempo observado.

⚠️ Esto **no** es la tasa anual de siniestros. Las pólizas tienen distinta `antiguedad`; la pregunta P2 modelará los **partes por año** con `offset(log(antiguedad))`.

In [ ]:
tabla_resumen <- datos_p1 %>%
  group_by(tipo_vehiculo, zona_circulacion) %>%
  summarise(
    sin_siniestro = sum(tuvo_siniestro == "No"),
    con_siniestro = sum(tuvo_siniestro == "Sí"),
    total = n(),
    .groups = "drop"
  ) %>%
  mutate(
    proporcion = con_siniestro / total,
    porcentaje = round(100 * proporcion, 1)
  ) %>%
  arrange(tipo_vehiculo, zona_circulacion)

stopifnot(sum(tabla_resumen$total) == nrow(datos_p1))

# Tabla de presentación: recuentos y porcentajes de cada grupo
print(tabla_resumen %>%
        select(tipo_vehiculo, zona_circulacion,
               sin_siniestro, con_siniestro, total, porcentaje),
      n = Inf)

## 3. ¿Hay mezcla de zonas? ☯

**La pregunta trampa del comité:** «Si las motos circularan sobre todo por ciudad…». Debemos **comprobarlo**, no asumirlo.

Primero, observamos la distribución de zonas *dentro* de cada tipo de vehículo. Después contrastamos el porcentaje de siniestro agregado por vehículo con el condicionado por zona. Si las composiciones difieren mucho, **comparar solo los porcentajes totales puede engañarnos**.

In [ ]:
composicion <- datos_p1 %>%
  count(tipo_vehiculo, zona_circulacion, name = "polizas") %>%
  group_by(tipo_vehiculo) %>%
  mutate(porcentaje_dentro_vehiculo = round(100 * polizas / sum(polizas), 1)) %>%
  ungroup()

marginal_vehiculo <- datos_p1 %>%
  group_by(tipo_vehiculo) %>%
  summarise(
    polizas = n(),
    con_siniestro = sum(tuvo_siniestro == "Sí"),
    porcentaje = round(100 * con_siniestro / polizas, 1),
    .groups = "drop"
  )

cat("Composición de las zonas en cada vehículo (%):\n")
print(composicion, n = Inf)
cat("\nPorcentaje agregado por tipo de vehículo (sin separar zonas):\n")
print(marginal_vehiculo, n = Inf)

## 4. Un mapa de calor para la presentación 🎨

Cada casilla es **una combinación vehículo–zona**, y el color representa la proporción de pólizas con siniestro. Incluimos `n` porque un porcentaje alto de un grupo pequeño es menos estable.

El gráfico permite ver la comparación en **una misma escala de color**; no debe sustituir a la tabla de recuentos.

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 6)

ggplot(tabla_resumen,
       aes(x = zona_circulacion, y = tipo_vehiculo, fill = proporcion)) +
  geom_tile(color = "white", linewidth = 1.1) +
  geom_text(
    aes(label = paste0(scales::percent(proporcion, accuracy = 0.1),
                       "\n(n = ", total, ")")),
    size = 5
  ) +
  scale_fill_gradient(
    low = "#fff9e6", high = "#d18f17",
    labels = scales::percent_format(accuracy = 1)
  ) +
  labs(
    title = "P1 · Pólizas con al menos un siniestro",
    subtitle = "Porcentaje dentro de cada combinación vehículo–zona (equipo 07)",
    x = "Zona de circulación", y = "Tipo de vehículo",
    fill = "Con siniestro"
  ) +
  theme_minimal(base_size = 15) +
  theme(panel.grid = element_blank())

## 5. ¿Qué pesa una vez controlado el otro factor? ⚙️

La tabla **describe** las diferencias; para valorar asociaciones *condicionadas* seguimos la **Unidad 2.2** con un modelo log-lineal Poisson de los **recuentos de las 18 celdas** (3 vehículos × 3 zonas × 2 estados).

Compararemos tres hipótesis:

- **Interacción triple:** ¿la asociación entre vehículo y siniestro cambia según la zona? Un modelo con todas las interacciones de dos vías se compara con el saturado mediante su desvianza.
- **Vehículo–siniestro:** una vez incluidos vehículo–zona y zona–siniestro, ¿falta la interacción vehículo–siniestro?
- **Zona–siniestro:** una vez incluidos vehículo–zona y vehículo–siniestro, ¿falta la interacción zona–siniestro?

Un `p` pequeño es **evidencia contra la restricción de independencia** correspondiente, no prueba de causalidad ni de importancia económica. Antes de interpretarlo comprobamos que las frecuencias ajustadas no sean diminutas.

In [ ]:
# Cada fila representa una celda (una combinación de tres factores).
frecuencias <- as.data.frame(tabla_3v) %>%
  rename(polizas = Freq)

modelo_pares <- glm(
  polizas ~ (tipo_vehiculo + zona_circulacion + tuvo_siniestro)^2,
  family = poisson(link = "log"), data = frecuencias
)
modelo_sin_vehiculo_evento <- glm(
  polizas ~ tipo_vehiculo * zona_circulacion +
             zona_circulacion * tuvo_siniestro,
  family = poisson(link = "log"), data = frecuencias
)
modelo_sin_zona_evento <- glm(
  polizas ~ tipo_vehiculo * zona_circulacion +
             tipo_vehiculo * tuvo_siniestro,
  family = poisson(link = "log"), data = frecuencias
)

# Estas comparaciones son pruebas de razón de verosimilitudes (χ² aproximado).
comparar_modelos <- function(reducido, completo) {
  prueba <- anova(reducido, completo, test = "Chisq")
  tibble(
    delta_desvianza = prueba$Deviance[2],
    gl = prueba$Df[2],
    valor_p = prueba[["Pr(>Chi)"]][2]
  )
}

pruebas <- bind_rows(
  tibble(
    hipotesis = "Interacción triple ausente",
    delta_desvianza = deviance(modelo_pares),
    gl = df.residual(modelo_pares),
    valor_p = pchisq(deviance(modelo_pares),
                     df.residual(modelo_pares),
                     lower.tail = FALSE)
  ),
  comparar_modelos(modelo_sin_vehiculo_evento, modelo_pares) %>%
    mutate(hipotesis = "Vehículo–siniestro ausente | zona"),
  comparar_modelos(modelo_sin_zona_evento, modelo_pares) %>%
    mutate(hipotesis = "Zona–siniestro ausente | vehículo")
) %>% select(hipotesis, delta_desvianza, gl, valor_p)

cat("Celdas observadas vacías:", sum(frecuencias$polizas == 0), "\n")
cat("Mínimo recuento esperado (modelo de pares):",
    round(min(fitted(modelo_pares)), 2), "\n")
cat("Celdas con esperado < 5:", sum(fitted(modelo_pares) < 5), "\n\n")
print(pruebas, n = Inf)

## 6. Interpretar antes de recomendar 🏵

**Lectura del equipo 07 (datos actuales):**

- Los porcentajes observados de **zona urbana** superan a los de mixta y rural **en los tres tipos de vehículo**.
- El contraste log-lineal de interacción triple no evidencia que la asociación vehículo–siniestro **cambie de forma sustancial** de una zona a otra (`p ≈ 0,734`).
- Dentro del modelo con interacciones de dos vías, **zona–siniestro** presenta una asociación clara tras incorporar vehículo (`p < 0,001`); **vehículo–siniestro** no aporta evidencia suficiente una vez incorporada la zona (`p ≈ 0,172`).

**Respuesta provisional para el comité:** parece más informativo **segmentar por zona de circulación** que basarse solo en el tipo de vehículo. No equivale a afirmar que el vehículo *nunca importa*; tampoco permite hablar de un efecto causal.

⚠️ **Límite crucial:** P1 usa `tuvo_siniestro` (probabilidad de haber tenido **algún** parte durante distintos tiempos de observación). Para decidir una **tarifa de frecuencia** hay que contrastar esta lectura con el modelo de **partes por año** de P2–P3: `n_siniestros` con `offset(log(antiguedad))`.

**Verificación:** ejecuta las celdas en orden y comprueba: 3.062 pólizas, 18 celdas, `sum(tabla_3v) == nrow(datos_p1)`, frecuencias esperadas suficientes y concordancia entre la tabla, el gráfico y los contrastes. Si cambia el CSV, revisa los números del texto antes de presentarlos.